## Data Reading

### CSV Files

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("Citibike Analysis").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/30 10:21:13 WARN Utils: Your hostname, pius-home, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/30 10:21:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/30 10:21:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
main_schema = """
    ride_id string, 
    rideable_type string,
    started_at timestamp,
    ended_at timestamp,
    start_station_name string, 
    start_station_id string,
    end_station_name string,
    end_station_id string,
    start_lat double,
    start_lng double,
    end_lat double,
    end_lng double,
    member_casual string
"""

trips = (
    spark.read.format("csv")
    .schema(main_schema)
    .option("header", True)
    .load("/home/piusm/dev-projects/analytics-engineering/Spark/data/JC*.csv")
)

26/06/30 10:21:17 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /home/piusm/dev-projects/analytics-engineering/Spark/data/JC*.csv.
java.io.FileNotFoundException: File /home/piusm/dev-projects/analytics-engineering/Spark/data/JC*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1Batch

In [4]:
print(f"Total rows loaded: {trips.count()}")

Total rows loaded: 4700530


In [5]:
# Or

# trips = spark.read.csv(
#     "/home/piusm/dev-projects/analytics-engineering/Spark/data/JC*.csv",
#     inferSchema=True,
#     header=True,
# )

#### Alternative way of defining a schema

In [6]:
# my_struct_schema = StructType(
#     [
#         StructField("ride_id", StringType(), True),
#         StructField("rideable_type", StringType(), True),
#         StructField("started_at", TimestampType(), True),
#         StructField("ended_at", TimestampType(), True),
#         StructField("start_station_name", StringType(), True),
#         StructField("start_station_id", StringType(), False),
#         StructField("end_station_name", StringType(), True),
#         StructField("end_station_id", StringType(), True),
#         StructField("start_lat", DoubleType(), True),
#         StructField("start_lng", DoubleType(), True),
#         StructField("end_lat", DoubleType(), True),
#         StructField("end_lng", DoubleType(), True),
#         StructField("member_casual", StringType(), True),
#     ]
# )

In [7]:
trips.show()

+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+
|         ride_id|rideable_type|          started_at|            ended_at|  start_station_name|start_station_id|    end_station_name|end_station_id|        start_lat|         start_lng|          end_lat|           end_lng|member_casual|
+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+
|172DBBFC733F03CE|electric_bike|2024-10-10 14:54:...|2024-10-10 15:04:...|         Oakland Ave|           JC022|Stevens - River T...|         HB602|       40.7376037|       -74.0524783|40.74313282710551|-74.02698867022991|       member|
|D20BBA4860FE736C|electric_bike|2024-10-03 19:20:...

In [8]:
trips.count()

4700530

In [9]:
trips.limit(10).toPandas()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,172DBBFC733F03CE,electric_bike,2024-10-10 14:54:24.572,2024-10-10 15:04:07.657,Oakland Ave,JC022,Stevens - River Ter & 6 St,HB602,40.737604,-74.052478,40.743133,-74.026989,member
1,D20BBA4860FE736C,electric_bike,2024-10-03 19:20:21.215,2024-10-03 19:31:46.511,Oakland Ave,JC022,Stevens - River Ter & 6 St,HB602,40.737604,-74.052478,40.743133,-74.026989,casual
2,86F89348995D0E6E,classic_bike,2024-10-20 12:14:56.318,2024-10-20 12:28:32.053,Oakland Ave,JC022,South Waterfront Walkway - Sinatra Dr & 1 St,HB103,40.737604,-74.052478,40.736982,-74.027781,casual
3,AA55A717B7EC1D10,classic_bike,2024-10-20 14:40:15.227,2024-10-20 14:55:39.100,Oakland Ave,JC022,Columbus Drive,JC014,40.737604,-74.052478,40.718355,-74.038914,member
4,C72953D91E986DA7,classic_bike,2024-10-20 08:37:03.280,2024-10-20 08:42:24.770,Brunswick & 6th,JC081,Washington St,JC098,40.726012,-74.050389,40.724294,-74.035483,member
5,23A1827EA03A9AC2,electric_bike,2024-10-28 19:20:28.668,2024-10-28 19:25:28.448,Oakland Ave,JC022,Hoboken Ave at Monmouth St,JC105,40.737604,-74.052478,40.735208,-74.046964,member
6,6C0E882AE20AC640,electric_bike,2024-10-08 10:41:37.926,2024-10-08 10:44:23.533,Pershing Field,JC024,Leonard Gordon Park,JC080,40.742677,-74.051789,40.745910,-74.057271,member
7,FC4AEE485D39016D,electric_bike,2024-10-25 21:02:40.878,2024-10-25 21:11:00.117,Pershing Field,JC024,Leonard Gordon Park,JC080,40.742677,-74.051789,40.745910,-74.057271,member
8,3E4D96936A8660C6,electric_bike,2024-10-08 12:17:23.948,2024-10-08 12:20:47.035,City Hall - Washington St & 1 St,HB105,Stevens - River Ter & 6 St,HB602,40.737360,-74.030970,40.743133,-74.026989,member
9,9F2FBA8132468A3B,electric_bike,2024-10-28 11:00:40.004,2024-10-28 11:03:12.823,City Hall - Washington St & 1 St,HB105,Stevens - River Ter & 6 St,HB602,40.737360,-74.030970,40.743133,-74.026989,member


### JSON Files

In [10]:
json_file = (
    spark.read.format("json")
    .option("inferSchema", True)
    .option("header", True)
    .option("multiLine", False)
    .load("/home/piusm/dev-projects/analytics-engineering/Spark/data/drivers.json")
)

In [11]:
json_file.limit(10).toPandas()

,code,dob,driverId,driverRef,name,nationality,number,url
0,HAM,1985-01-07,1,hamilton,"(Lewis, Hamilton)",British,44,http://en.wikipedia.org/wiki/Lewis_Hamilton
1,HEI,1977-05-10,2,heidfeld,"(Nick, Heidfeld)",German,\N,http://en.wikipedia.org/wiki/Nick_Heidfeld
2,ROS,1985-06-27,3,rosberg,"(Nico, Rosberg)",German,6,http://en.wikipedia.org/wiki/Nico_Rosberg
3,ALO,1981-07-29,4,alonso,"(Fernando, Alonso)",Spanish,14,http://en.wikipedia.org/wiki/Fernando_Alonso
4,KOV,1981-10-19,5,kovalainen,"(Heikki, Kovalainen)",Finnish,\N,http://en.wikipedia.org/wiki/Heikki_Kovalainen
5,NAK,1985-01-11,6,nakajima,"(Kazuki, Nakajima)",Japanese,\N,http://en.wikipedia.org/wiki/Kazuki_Nakajima
6,BOU,1979-02-28,7,bourdais,"(Sébastien, Bourdais)",French,\N,http://en.wikipedia.org/wiki/S%C3%A9bastien_Bo...
7,RAI,1979-10-17,8,raikkonen,"(Kimi, Räikkönen)",Finnish,7,http://en.wikipedia.org/wiki/Kimi_R%C3%A4ikk%C...
8,KUB,1984-12-07,9,kubica,"(Robert, Kubica)",Polish,88,http://en.wikipedia.org/wiki/Robert_Kubica
9,GLO,1982-03-18,10,glock,"(Timo, Glock)",German,\N,http://en.wikipedia.org/wiki/Timo_Glock


### Schema Definition

In [12]:
trips.printSchema()

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)



In [13]:
trips.printSchema()

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)



#### StructType() - Programmatic way to define schema

In [14]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

### Data Transformation with PySpark

In [15]:
trips.select("ride_id", "rideable_type").show()

+----------------+-------------+
|         ride_id|rideable_type|
+----------------+-------------+
|172DBBFC733F03CE|electric_bike|
|D20BBA4860FE736C|electric_bike|
|86F89348995D0E6E| classic_bike|
|AA55A717B7EC1D10| classic_bike|
|C72953D91E986DA7| classic_bike|
|23A1827EA03A9AC2|electric_bike|
|6C0E882AE20AC640|electric_bike|
|FC4AEE485D39016D|electric_bike|
|3E4D96936A8660C6|electric_bike|
|9F2FBA8132468A3B|electric_bike|
|B71B07C28733404F|electric_bike|
|2EE1AB7AFEFB4CBB| classic_bike|
|A7493DA7AFDDBC44| classic_bike|
|7A956542D29EA30E| classic_bike|
|157EFB12EA94DA9A|electric_bike|
|3A54C7F77E03FB73| classic_bike|
|94374385A5951BEE| classic_bike|
|5E9040B2EF62AE11|electric_bike|
|99D204D580A9BD0C|electric_bike|
|7C3E2DA95C52FDFB|electric_bike|
+----------------+-------------+
only showing top 20 rows


In [16]:
trips.select(col("ride_id"), col("rideable_type")).show()

+----------------+-------------+
|         ride_id|rideable_type|
+----------------+-------------+
|172DBBFC733F03CE|electric_bike|
|D20BBA4860FE736C|electric_bike|
|86F89348995D0E6E| classic_bike|
|AA55A717B7EC1D10| classic_bike|
|C72953D91E986DA7| classic_bike|
|23A1827EA03A9AC2|electric_bike|
|6C0E882AE20AC640|electric_bike|
|FC4AEE485D39016D|electric_bike|
|3E4D96936A8660C6|electric_bike|
|9F2FBA8132468A3B|electric_bike|
|B71B07C28733404F|electric_bike|
|2EE1AB7AFEFB4CBB| classic_bike|
|A7493DA7AFDDBC44| classic_bike|
|7A956542D29EA30E| classic_bike|
|157EFB12EA94DA9A|electric_bike|
|3A54C7F77E03FB73| classic_bike|
|94374385A5951BEE| classic_bike|
|5E9040B2EF62AE11|electric_bike|
|99D204D580A9BD0C|electric_bike|
|7C3E2DA95C52FDFB|electric_bike|
+----------------+-------------+
only showing top 20 rows


#### Using ALIAS

In [17]:
trips.select(col("ride_id").alias("rideId")).show()

+----------------+
|          rideId|
+----------------+
|172DBBFC733F03CE|
|D20BBA4860FE736C|
|86F89348995D0E6E|
|AA55A717B7EC1D10|
|C72953D91E986DA7|
|23A1827EA03A9AC2|
|6C0E882AE20AC640|
|FC4AEE485D39016D|
|3E4D96936A8660C6|
|9F2FBA8132468A3B|
|B71B07C28733404F|
|2EE1AB7AFEFB4CBB|
|A7493DA7AFDDBC44|
|7A956542D29EA30E|
|157EFB12EA94DA9A|
|3A54C7F77E03FB73|
|94374385A5951BEE|
|5E9040B2EF62AE11|
|99D204D580A9BD0C|
|7C3E2DA95C52FDFB|
+----------------+
only showing top 20 rows


### Filtering Data

#### Scenario 1

In [18]:
trips.filter(col("member_casual") == "casual").show()

+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+------------------+------------------+-----------------+------------------+-------------+
|         ride_id|rideable_type|          started_at|            ended_at|  start_station_name|start_station_id|    end_station_name|end_station_id|         start_lat|         start_lng|          end_lat|           end_lng|member_casual|
+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+------------------+------------------+-----------------+------------------+-------------+
|D20BBA4860FE736C|electric_bike|2024-10-03 19:20:...|2024-10-03 19:31:...|         Oakland Ave|           JC022|Stevens - River T...|         HB602|        40.7376037|       -74.0524783|40.74313282710551|-74.02698867022991|       casual|
|86F89348995D0E6E| classic_bike|2024-10-20 12:14

#### Scenario 2


In [19]:
trips.filter(
    (col("member_casual") == "casual") & (col("rideable_type") == "docked_bike")
).show()

+----------------+-------------+-------------------+-------------------+--------------------+----------------+--------------------+--------------+---------+----------+---------+----------+-------------+
|         ride_id|rideable_type|         started_at|           ended_at|  start_station_name|start_station_id|    end_station_name|end_station_id|start_lat| start_lng|  end_lat|   end_lng|member_casual|
+----------------+-------------+-------------------+-------------------+--------------------+----------------+--------------------+--------------+---------+----------+---------+----------+-------------+
|5E8C8B7DFFA9C73E|  docked_bike|2022-08-20 01:17:27|2022-08-20 02:01:02|South Waterfront ...|           HB103|Hoboken Terminal ...|         HB101|40.736982|-74.027781|40.735938|-74.030305|       casual|
|0A0EFF1E90C8710F|  docked_bike|2022-08-06 20:06:16|2022-08-06 20:18:10|South Waterfront ...|           HB103|Mama Johnson Fiel...|         HB404|40.736982|-74.027781| 40.74314|-74.040041|

#### Scenario 3

In [20]:
trips.filter((col("ride_id").isNull())).show()

+-------+-------------+----------+--------+------------------+----------------+----------------+--------------+---------+---------+-------+-------+-------------+
|ride_id|rideable_type|started_at|ended_at|start_station_name|start_station_id|end_station_name|end_station_id|start_lat|start_lng|end_lat|end_lng|member_casual|
+-------+-------------+----------+--------+------------------+----------------+----------------+--------------+---------+---------+-------+-------+-------------+
+-------+-------------+----------+--------+------------------+----------------+----------------+--------------+---------+---------+-------+-------+-------------+



In [21]:
trips.filter((col("start_station_id").isNull()) & (col("ride_id").isNull())).show()

+-------+-------------+----------+--------+------------------+----------------+----------------+--------------+---------+---------+-------+-------+-------------+
|ride_id|rideable_type|started_at|ended_at|start_station_name|start_station_id|end_station_name|end_station_id|start_lat|start_lng|end_lat|end_lng|member_casual|
+-------+-------------+----------+--------+------------------+----------------+----------------+--------------+---------+---------+-------+-------+-------------+
+-------+-------------+----------+--------+------------------+----------------+----------------+--------------+---------+---------+-------+-------+-------------+



In [22]:
trips.where((col("start_station_name") == "Warren St")).show()

+----------------+-------------+--------------------+--------------------+------------------+----------------+--------------------+--------------+----------+------------+-----------------+------------------+-------------+
|         ride_id|rideable_type|          started_at|            ended_at|start_station_name|start_station_id|    end_station_name|end_station_id| start_lat|   start_lng|          end_lat|           end_lng|member_casual|
+----------------+-------------+--------------------+--------------------+------------------+----------------+--------------------+--------------+----------+------------+-----------------+------------------+-------------+
|143732490A22F95A| classic_bike|2024-10-26 09:54:...|2024-10-26 10:12:...|         Warren St|           JC006|Stevens - River T...|         HB602|40.7211236|-74.03805095|40.74313282710551|-74.02698867022991|       member|
|23BDC5B0B86C99B4| classic_bike|2024-10-26 10:08:...|2024-10-26 10:49:...|         Warren St|           JC006|St

#### Dropping values based on a condition

In [23]:
trips = trips.dropna(subset=["ride_id", "start_lat", "end_lat", "start_lng", "end_lng"])

In [24]:
trips.count()

4689909

#### withColumnRenamed()

In [25]:
trips.withColumnRenamed("ride_id", "rideId").show()

+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+
|          rideId|rideable_type|          started_at|            ended_at|  start_station_name|start_station_id|    end_station_name|end_station_id|        start_lat|         start_lng|          end_lat|           end_lng|member_casual|
+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+
|172DBBFC733F03CE|electric_bike|2024-10-10 14:54:...|2024-10-10 15:04:...|         Oakland Ave|           JC022|Stevens - River T...|         HB602|       40.7376037|       -74.0524783|40.74313282710551|-74.02698867022991|       member|
|D20BBA4860FE736C|electric_bike|2024-10-03 19:20:...

#### withColumn() - Adding a new column to the DataFrame

In [26]:
trips = trips.withColumn(
    "rideDurationMinutes",
    round((col("ended_at").cast("long") - col("started_at").cast("long")) / 60, 2),
)

trips = trips.withColumn(
    "rideDurationMinutes",
    when(col("rideDurationMinutes") < 0, col("rideDurationMinutes") + 60).otherwise(
        col("rideDurationMinutes")
    ),
)

trips = trips.withColumn("rideDurationHours", round(col("rideDurationMinutes") / 60, 2))

In [27]:
trips.show()

+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+
|         ride_id|rideable_type|          started_at|            ended_at|  start_station_name|start_station_id|    end_station_name|end_station_id|        start_lat|         start_lng|          end_lat|           end_lng|member_casual|rideDurationMinutes|rideDurationHours|
+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+
|172DBBFC733F03CE|electric_bike|2024-10-10 14:54:...|2024-10-10 15:04:...|         Oakland Ave|           JC022|Stevens - River T...|         HB602|       40.7376037|       -7

In [28]:
R = 6371.0

lat1 = radians(col("start_lat"))
lat2 = radians(col("end_lat"))
lng1 = radians(col("start_lng"))
lng2 = radians(col("end_lng"))

dlat = lat2 - lat1
dlng = lng2 - lng1

a = pow(sin(dlat / 2.0), 2) + cos(lat1) * cos(lat2) * pow(sin(dlng / 2.0), 2)

c = 2 * asin(sqrt(a))

distance_formula = round(R * c, 2)

trips = trips.withColumn("rideDistanceKM", distance_formula).withColumn(
    "rideDistanceM", distance_formula * 1000
)

In [29]:
trips.printSchema()

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)
 |-- rideDurationMinutes: double (nullable = true)
 |-- rideDurationHours: double (nullable = true)
 |-- rideDistanceKM: double (nullable = true)
 |-- rideDistanceM: double (nullable = true)



In [30]:
trips.filter((col("rideDistanceM") == 0)).show()

+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|         ride_id|rideable_type|          started_at|            ended_at|  start_station_name|start_station_id|    end_station_name|end_station_id|        start_lat|         start_lng|          end_lat|           end_lng|member_casual|rideDurationMinutes|rideDurationHours|rideDistanceKM|rideDistanceM|
+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|E12C216DABC5E494| classic_bike|2024-10-31 11:38:...|2024-10-31 11:41:...|       Washing

In [31]:
trips.filter((col("rideDistanceM") == 0)).count()

261413

#### Type Casting

In [32]:
trips.withColumn("ride_id", col("ride_id").cast(StringType()))

DataFrame[ride_id: string, rideable_type: string, started_at: timestamp, ended_at: timestamp, start_station_name: string, start_station_id: string, end_station_name: string, end_station_id: string, start_lat: double, start_lng: double, end_lat: double, end_lng: double, member_casual: string, rideDurationMinutes: double, rideDurationHours: double, rideDistanceKM: double, rideDistanceM: double]

#### Sort and OrderBy

In [33]:
trips.sort((col("rideDurationMinutes").desc())).show()

+----------------+-------------+-------------------+-------------------+--------------------+----------------+--------------------+--------------+---------+----------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|         ride_id|rideable_type|         started_at|           ended_at|  start_station_name|start_station_id|    end_station_name|end_station_id|start_lat| start_lng|          end_lat|           end_lng|member_casual|rideDurationMinutes|rideDurationHours|rideDistanceKM|rideDistanceM|
+----------------+-------------+-------------------+-------------------+--------------------+----------------+--------------------+--------------+---------+----------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|AFF1F450B44EF59E|  docked_bike|2021-07-31 19:36:50|2021-12-08 16:38:19|South Waterfront ...|           HB103|Grand Concourse &...|       8303

In [34]:
trips.sort((col("rideDurationMinutes").asc())).show()

+----------------+-------------+-------------------+-------------------+--------------------+----------------+--------------------+--------------+------------------+------------------+------------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|         ride_id|rideable_type|         started_at|           ended_at|  start_station_name|start_station_id|    end_station_name|end_station_id|         start_lat|         start_lng|           end_lat|           end_lng|member_casual|rideDurationMinutes|rideDurationHours|rideDistanceKM|rideDistanceM|
+----------------+-------------+-------------------+-------------------+--------------------+----------------+--------------------+--------------+------------------+------------------+------------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|FF9A083AF6A76137| classic_bike|2023-08-22 18:41:21|2023-08-22 18:41:21|      Pershing F

In [35]:
trips.sort(
    ["rideDurationMinutes", "rideDistanceKM"], ascending=[0, 0]
).show()  # 0, 0 means ascending = False - boolean

+----------------+-------------+-------------------+-------------------+--------------------+----------------+--------------------+--------------+---------+----------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|         ride_id|rideable_type|         started_at|           ended_at|  start_station_name|start_station_id|    end_station_name|end_station_id|start_lat| start_lng|          end_lat|           end_lng|member_casual|rideDurationMinutes|rideDurationHours|rideDistanceKM|rideDistanceM|
+----------------+-------------+-------------------+-------------------+--------------------+----------------+--------------------+--------------+---------+----------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|AFF1F450B44EF59E|  docked_bike|2021-07-31 19:36:50|2021-12-08 16:38:19|South Waterfront ...|           HB103|Grand Concourse &...|       8303

In [36]:
trips.sort(["rideDurationMinutes", "rideDistanceKM"], ascending=[0, 1]).show()

+----------------+-------------+-------------------+-------------------+--------------------+----------------+--------------------+--------------+---------+----------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|         ride_id|rideable_type|         started_at|           ended_at|  start_station_name|start_station_id|    end_station_name|end_station_id|start_lat| start_lng|          end_lat|           end_lng|member_casual|rideDurationMinutes|rideDurationHours|rideDistanceKM|rideDistanceM|
+----------------+-------------+-------------------+-------------------+--------------------+----------------+--------------------+--------------+---------+----------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|AFF1F450B44EF59E|  docked_bike|2021-07-31 19:36:50|2021-12-08 16:38:19|South Waterfront ...|           HB103|Grand Concourse &...|       8303

#### Limit